# Hierarchical clustering of human saliva

Figure 3a. Ward clustering and stacked family-level relative abundance profiles.

In [ ]:
from pathlib import Path

# Jupyter normally starts the kernel in the notebook directory.
here = Path.cwd()
repo_root = next(parent for parent in (here, *here.parents)
                 if (parent / "requirements.txt").is_file()
                 and (parent / "mouse").is_dir() and (parent / "human").is_dir())
data_dir = repo_root / "data"
output_dir = repo_root / "outputs" / 'human/01_saliva_dendrogram'
output_dir.mkdir(parents=True, exist_ok=True)
notebook_output = output_dir


## Load the abundance workbook and identify metadata

In [ ]:
import pandas as pd

# Define the file path
path = data_dir / 'Human_RawData_for_PCA_SALIVA.xlsx'

# Read the Excel file into a DataFrame
df = pd.read_excel(path)
# Remove rows where all values are NaN
df = df.dropna(how='all')

df


In [ ]:
# Define label columns (metadata)
label_cols = ['SampleID', 'Age', 'Gender', 'Treatment', 'Disease_Duration', 'Symptom_Duration', 'H_Y_Score', 'Collection']

# Define feature columns (everything else)
feature_cols = [col for col in df.columns if col not in label_cols]

# Create separate DataFrames
df_labels = df[label_cols]
df_features = df[feature_cols]


## Cluster samples and plot composition

In [ ]:
# ============================================================
# Hierarchical Clustering with Two Metadata Strips
# (Group / Sex) + Stacked Family Abundance Bars
# ============================================================

import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
import numpy as np
import pandas as pd

# -------------------------------
# Global font and style settings
# -------------------------------
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 14,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 20
})
plt.rcParams['pdf.fonttype'] = 42      # TrueType for vector output
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'  # Keep text editable

# -------------------------------
# 1. Prepare labels
# Rename to consistent names expected by plotting
# -------------------------------
df_labels = df_labels.rename(columns={
    'Treatment': 'Group',
    'Gender': 'Sex'
})

# Optional: composite label for leaves
df_labels['Label'] = df_labels['Sex'] + "_" + df_labels['Group']

# -------------------------------
# 2. Keep up to top 16 families, group others into "Other", normalize
# -------------------------------
df_feat_numeric = df_features.select_dtypes(include=[np.number])
mean_abundance = df_feat_numeric.mean().sort_values(ascending=False)
print(f"Found {len(mean_abundance)} bacterial families in dataset.")

top_n = min(16, len(mean_abundance) - 1) if len(mean_abundance) > 1 else len(mean_abundance)
top_families = mean_abundance.index[:top_n]
other_families = mean_abundance.index[top_n:]

if len(other_families) > 0:
    df_features_filtered = df_feat_numeric[top_families].copy()
    df_features_filtered['Other'] = df_feat_numeric[other_families].sum(axis=1)
else:
    df_features_filtered = df_feat_numeric[top_families].copy()

# Normalize rows to sum = 1
df_features_filtered = df_features_filtered.div(df_features_filtered.sum(axis=1), axis=0)
print(f"Final plotted families: {len(df_features_filtered.columns)} → {list(df_features_filtered.columns)}")

# -------------------------------
# 3. Hierarchical clustering
# -------------------------------
linkage_matrix = linkage(df_features_filtered, method='ward')

# -------------------------------
# 4. Figure setup (2 metadata strips)
# -------------------------------
fig = plt.figure(figsize=(20, 13))
gs = fig.add_gridspec(4, 1, height_ratios=[1, 0.05, 0.05, 2], hspace=0.05)
ax_dendro = fig.add_subplot(gs[0])
ax_strip_sex = fig.add_subplot(gs[1], sharex=ax_dendro)
ax_strip_group = fig.add_subplot(gs[2], sharex=ax_dendro)
ax_bar = fig.add_subplot(gs[3], sharex=ax_dendro)

# -------------------------------
# 5. Dendrogram
# -------------------------------
dendro = dendrogram(
    linkage_matrix,
    labels=df_labels['Label'].values,
    leaf_rotation=90,
    leaf_font_size=8,
    ax=ax_dendro,
    color_threshold=0,
    link_color_func=lambda k: "black"
)

ax_dendro.set_xticks([])
ax_dendro.set_xticklabels([])
ax_dendro.set_ylabel('Distance')
ax_dendro.set_title(
    "Hierarchical Clustering of Bacterial Relative Abundance",
    fontweight='bold'
)

# -------------------------------
# 6. Reorder data based on dendrogram leaves
# -------------------------------
ordered_idx = dendro['leaves']
ordered_features = df_features_filtered.iloc[ordered_idx]
ordered_labels = df_labels.iloc[ordered_idx]

# -------------------------------
# 7. Align bar chart and strips
# -------------------------------
x_start, x_end = ax_dendro.get_xlim()
x = np.linspace(x_start, x_end, len(ordered_idx))
bar_width = (x[1] - x[0]) * 0.8

for ax in [ax_strip_sex, ax_strip_group, ax_bar]:
    ax.set_xticks([])
    ax.set_xticklabels([])

# -------------------------------
# 8. Metadata strips (custom color maps)
# -------------------------------
group_colors_map = {"Control": "#42A5F5", "Parkinson": "#EF5350"}  # update as needed
sex_colors_map = {"Male": "lightskyblue", "Female": "magenta"}

group_colors = ordered_labels['Group'].map(group_colors_map).fillna("lightgray")
sex_colors = ordered_labels['Sex'].map(sex_colors_map).fillna("lightgray")

ax_strip_group.bar(x, [1]*len(x), color=group_colors, width=bar_width, align='center')
ax_strip_sex.bar(x, [1]*len(x), color=sex_colors, width=bar_width, align='center')

for ax in [ax_strip_sex, ax_strip_group]:
    ax.set_yticks([])
    ax.set_ylim(0, 1.05)

# -------------------------------
# 9. Add strip labels
# -------------------------------
x_min, x_max = ax_dendro.get_xlim()
x_right = x_max + (x_max - x_min) * 0.02  # position to the right

ax_strip_group.text(x_right, 0.5, 'Group', va='center', ha='left',
                    fontsize=14, fontweight='bold')
ax_strip_sex.text(x_right, 0.5, 'Sex', va='center', ha='left',
                  fontsize=14, fontweight='bold')

# -------------------------------
# 10. Stacked bar chart (bacterial families)
# -------------------------------
bottoms = np.zeros(len(ordered_features))
colors = plt.cm.get_cmap('tab20', len(ordered_features.columns)).colors

for i, col in enumerate(ordered_features.columns):
    ax_bar.bar(
        x,
        ordered_features[col],
        bottom=bottoms,
        color=colors[i % len(colors)],
        width=bar_width,
        label=col,
        align='center'
    )
    bottoms += ordered_features[col]

ax_bar.set_ylabel("Relative Abundance", fontsize=16, fontweight='bold')

# Expand x‐limits slightly
x_min, x_max = ax_dendro.get_xlim()
expand = (x_max - x_min) / len(ordered_idx) / 2
for ax in [ax_strip_sex, ax_strip_group, ax_bar]:
    ax.set_xlim(x_min - expand, x_max + expand)

# -------------------------------
# 11. Legend (families only)
# -------------------------------
family_handles = [
    plt.Rectangle((0, 0), 1, 1, color=colors[i % len(colors)])
    for i in range(len(ordered_features.columns))
]

fig.legend(
    family_handles, ordered_features.columns, title="Bacterial Family",
    loc='center right', bbox_to_anchor=(1.02, 0.4),
    fontsize=12, title_fontsize=14, frameon=False
)

plt.tight_layout()

# Optional: save high‑resolution vector output
output_path = notebook_output
plt.savefig(output_path / "Human_SalivaDendrogram.pdf", format="pdf", bbox_inches="tight")

plt.show()
